# Pipeline trenowania modelu – półmaraton Wrocław 2023

**Cel:** oszacować czas ukończenia półmaratonu na podstawie danych, które poda użytkownik w aplikacji: **płeć, wiek i czas na 5 km**.

Kroki pipeline'u:
1. Wczytanie danych z Digital Ocean Spaces (z fallbackiem do pliku lokalnego)
2. Czyszczenie danych
3. Wybór cech (feature selection) i porównanie modeli na wybranych metrykach
4. Trening finalnego modelu
5. Zapis nowej wersji modelu lokalnie **i** do Digital Ocean Spaces

> Czasy na 10/15/20 km oraz tempo i miejsca są **świadomie pominięte** – użytkownik ich nie poda, a ich użycie oznaczałoby *data leakage* (znamy je dopiero w trakcie biegu).

## 1. Konfiguracja i importy

In [ ]:
import glob, io, json, os, re, warnings
from datetime import datetime

import boto3
import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             median_absolute_error, r2_score, root_mean_squared_error)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
load_dotenv()  # wczytuje zmienne z pliku .env (klucze Digital Ocean Spaces)

RACE_YEAR = 2023
RANDOM_STATE = 42

SPACES_KEY = os.getenv("DO_SPACES_KEY")
SPACES_SECRET = os.getenv("DO_SPACES_SECRET")
SPACES_REGION = os.getenv("DO_SPACES_REGION", "sfo3")
SPACES_BUCKET = os.getenv("DO_SPACES_BUCKET")

DATA_PREFIX = "data/"          # folder w Spaces z plikami (CSV / XLSX), jeden plik = jeden bieg
MODEL_PREFIX = "models/"
LOCAL_DATA_DIR = "data"
LOCAL_MODEL_DIR = "model"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

SPACES_ENABLED = all([SPACES_KEY, SPACES_SECRET, SPACES_BUCKET])
print("Digital Ocean Spaces:", "włączone" if SPACES_ENABLED else "BRAK KLUCZY – tryb lokalny")

## 2. Wczytanie danych z Digital Ocean Spaces

In [ ]:
def get_s3_client():
    return boto3.session.Session().client(
        "s3",
        region_name=SPACES_REGION,
        endpoint_url=f"https://{SPACES_REGION}.digitaloceanspaces.com",
        aws_access_key_id=SPACES_KEY,
        aws_secret_access_key=SPACES_SECRET,
    )


SUPPORTED = (".csv", ".xlsx", ".xls")


def read_any(body: bytes, name: str) -> pd.DataFrame:
    """CSV (separator ;) lub Excel."""
    if name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(body))
    return pd.read_csv(io.BytesIO(body), sep=";")


def race_year_from_name(name: str) -> int:
    """Rok biegu z nazwy pliku, np. halfmarathon_wroclaw_2024__final.csv -> 2024."""
    m = re.search(r"(20\d{2})", os.path.basename(name))
    if not m:
        raise ValueError(f"Brak roku (20xx) w nazwie pliku: {name}")
    return int(m.group(1))


def load_raw_data() -> pd.DataFrame:
    frames = []
    if SPACES_ENABLED:
        s3 = get_s3_client()
        sources = []
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=SPACES_BUCKET, Prefix=DATA_PREFIX):
            sources += [o["Key"] for o in page.get("Contents", []) if o["Key"].lower().endswith(SUPPORTED)]
        reader = lambda key: s3.get_object(Bucket=SPACES_BUCKET, Key=key)["Body"].read()
    else:
        sources = [p for p in sorted(glob.glob(f"{LOCAL_DATA_DIR}/*")) if p.lower().endswith(SUPPORTED)]
        reader = lambda path: open(path, "rb").read()

    for src in sources:
        d = read_any(reader(src), src)
        d["rok_biegu"] = race_year_from_name(src)
        d["zrodlo"] = src
        frames.append(d)
        print(f"Wczytano {src}: {len(d)} wierszy (rok {d['rok_biegu'].iloc[0]})")

    if not frames:
        raise FileNotFoundError("Nie znaleziono żadnych plików z danymi (.csv/.xlsx).")
    return pd.concat(frames, ignore_index=True)


df_raw = load_raw_data()
print("Razem:", df_raw.shape)
df_raw.groupby("rok_biegu").size()

## 3. Czyszczenie danych

Zasady:
* usuwamy osoby bez czasu końcowego (`DNS`/`DNF`) lub bez czasu na 5 km,
* usuwamy wiersze bez płci,
* wiek = `2023 - Rocznik`; błędne roczniki (np. `0`) uzupełniamy środkiem kategorii wiekowej (np. `M30` → 35 lat), a gdy się nie da – usuwamy wiersz,
* usuwamy nierealne wartości i outliery (np. czas 5 km wolniejszy niż tempo mety, błędy pomiaru).

In [ ]:
def convert_time_to_seconds(time):
    if pd.isnull(time) or time in ["DNS", "DNF"]:
        return np.nan
    h, m, s = str(time).split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


def age_from_row(row):
    year = row["rok_biegu"]
    rocznik = row["Rocznik"]
    if pd.notnull(rocznik) and year - 90 <= rocznik <= year - 12:
        return year - int(rocznik)
    cat = row["Kategoria wiekowa"]
    if pd.notnull(cat) and str(cat)[1:].isdigit():
        return int(str(cat)[1:]) + 5
    return np.nan


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["czas_s"] = df["Czas"].apply(convert_time_to_seconds)
    out["czas_5km_s"] = df["5 km Czas"].apply(convert_time_to_seconds)
    out["plec"] = df["Płeć"]
    out["wiek"] = df.apply(age_from_row, axis=1)

    n0 = len(out)
    out = out.dropna()
    print(f"Usunięto braki danych: {n0 - len(out)}")

    n1 = len(out)
    ratio = out["czas_s"] / out["czas_5km_s"]
    mask = (
        out["plec"].isin(["M", "K"])
        & out["czas_5km_s"].between(12 * 60, 60 * 60)        # 5 km: 12–60 min
        & out["czas_s"].between(55 * 60, 4 * 3600)           # półmaraton: 55 min – 4 h
        & ratio.between(3.5, 6.0)                            # sensowny stosunek mety do 5 km
        & out["wiek"].between(14, 90)
    )
    out = out[mask].reset_index(drop=True)
    print(f"Usunięto outliery / wartości nierealne: {n1 - len(out)}")
    print(f"Zostało {len(out)} z {len(df)} wierszy")
    return out


df = clean_data(df_raw)
df.describe()

## 4. Feature selection i wybór metryk

### Proponowane metryki (regresja czasu w sekundach)
| Metryka | Dlaczego |
|---|---|
| **MAE** (średni błąd bezwzględny) | główna metryka – wynik w sekundach/minutach, łatwy do zakomunikowania użytkownikowi („myli się średnio o X min”) |
| **RMSE** | mocniej karze duże pomyłki, pokazuje czy model nie ma „wielkich wpadek” |
| **MedAE** (mediana błędu) | odporna na outliery – typowy błąd |
| **MAPE** | błąd procentowy – porównywalny dla szybkich i wolnych biegaczy |
| **R²** | jaka część zmienności czasu mety jest wyjaśniona przez model |

Wybór modelu: **najniższe MAE w walidacji krzyżowej** (5-fold).

### Kandydaci na cechy
`czas_5km_s`, `wiek`, `plec`. Sprawdzamy je trzema sposobami: korelacją / informacją wzajemną, ważnością permutacyjną i porównaniem zestawów cech.

In [ ]:
X_all = df[["czas_5km_s", "wiek", "plec"]].copy()
y = df["czas_s"]

# a) korelacja i mutual information
mi = mutual_info_regression(
    X_all.assign(plec=(X_all["plec"] == "M").astype(int)), y, random_state=RANDOM_STATE
)
print("Korelacja z czasem mety:")
print(df.assign(plec_M=(df["plec"] == "M").astype(int))[["czas_5km_s", "wiek", "plec_M", "czas_s"]].corr()["czas_s"].round(3))
print("\nMutual information:", dict(zip(X_all.columns, mi.round(3))))

In [ ]:
def make_pipeline(model, features):
    cat = [f for f in features if f == "plec"]
    num = [f for f in features if f != "plec"]
    pre = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat),
    ])
    return Pipeline([("prep", pre), ("model", model)])


SCORING = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "MedAE": "neg_median_absolute_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2",
}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# b) porównanie zestawów cech (ten sam model – GradientBoosting)
feature_sets = {
    "5km": ["czas_5km_s"],
    "5km + plec": ["czas_5km_s", "plec"],
    "5km + wiek": ["czas_5km_s", "wiek"],
    "5km + plec + wiek": ["czas_5km_s", "plec", "wiek"],
}
rows = []
for name, feats in feature_sets.items():
    res = cross_validate(
        make_pipeline(GradientBoostingRegressor(random_state=RANDOM_STATE), feats),
        X_all[feats], y, cv=cv, scoring=SCORING)
    rows.append({"zestaw cech": name,
                 "MAE [s]": -res["test_MAE"].mean(),
                 "RMSE [s]": -res["test_RMSE"].mean(),
                 "MAPE [%]": -res["test_MAPE"].mean() * 100,
                 "R2": res["test_R2"].mean()})
pd.DataFrame(rows).round(2).set_index("zestaw cech")

Na podstawie powyższej tabeli wybieramy zestaw cech z najniższym MAE (poniżej – domyślnie pełny zestaw, który odpowiada temu, co poda użytkownik).

In [ ]:
FEATURES = ["czas_5km_s", "plec", "wiek"]   # zmień, jeśli tabela wyżej wskaże inny zestaw

# podział train / test (test używany TYLKO do końcowej oceny)
X_train, X_test, y_train, y_test = train_test_split(
    X_all[FEATURES], y, test_size=0.2, random_state=RANDOM_STATE)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
for name, model in models.items():
    res = cross_validate(make_pipeline(model, FEATURES), X_train, y_train, cv=cv, scoring=SCORING)
    results.append({"model": name,
                    "MAE [s]": -res["test_MAE"].mean(),
                    "RMSE [s]": -res["test_RMSE"].mean(),
                    "MedAE [s]": -res["test_MedAE"].mean(),
                    "MAPE [%]": -res["test_MAPE"].mean() * 100,
                    "R2": res["test_R2"].mean()})
results_df = pd.DataFrame(results).set_index("model").sort_values("MAE [s]")
display(results_df.round(3))

best_name = results_df.index[0]
print("Najlepszy model (CV MAE):", best_name)

### Ważność cech (permutation importance) dla najlepszego modelu

In [ ]:
best_pipeline = make_pipeline(models[best_name], FEATURES).fit(X_train, y_train)
pi = permutation_importance(best_pipeline, X_test, y_test, scoring="neg_mean_absolute_error",
                            n_repeats=10, random_state=RANDOM_STATE)
pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(1).rename("wzrost MAE [s] po przetasowaniu cechy")

## 5. Ocena finalna na zbiorze testowym i trening na całości danych

In [ ]:
pred = best_pipeline.predict(X_test)
test_metrics = {
    "MAE_s": float(mean_absolute_error(y_test, pred)),
    "RMSE_s": float(root_mean_squared_error(y_test, pred)),
    "MedAE_s": float(median_absolute_error(y_test, pred)),
    "MAPE_pct": float(mean_absolute_percentage_error(y_test, pred) * 100),
    "R2": float(r2_score(y_test, pred)),
}
print({k: round(v, 3) for k, v in test_metrics.items()})
print(f"Średni błąd: {test_metrics['MAE_s']/60:.1f} min")

# finalny model trenowany na wszystkich oczyszczonych danych
final_model = make_pipeline(models[best_name], FEATURES).fit(X_all[FEATURES], y)

## 6. Zapis modelu – lokalnie i do Digital Ocean Spaces

Każdy trening tworzy **nową wersję** (`half_marathon_model_<timestamp>.joblib`) oraz aktualizuje wskaźnik `latest.joblib`, z którego korzysta aplikacja.

In [ ]:
version = datetime.now().strftime("%Y%m%d_%H%M%S")
import sklearn
metadata = {
    "version": version,
    "model_name": best_name,
    "features": FEATURES,
    "n_samples": int(len(df)),
    "test_metrics": test_metrics,
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
}

versioned_name = f"half_marathon_model_{version}.joblib"
artifact = {"model": final_model, "metadata": metadata}

# --- lokalnie ---
for fname in (versioned_name, "latest.joblib"):
    joblib.dump(artifact, os.path.join(LOCAL_MODEL_DIR, fname))
with open(os.path.join(LOCAL_MODEL_DIR, "latest_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Zapisano lokalnie w:", LOCAL_MODEL_DIR)

# --- Digital Ocean Spaces ---
if SPACES_ENABLED:
    s3 = get_s3_client()
    for fname in (versioned_name, "latest.joblib", "latest_metadata.json"):
        key = f"{MODEL_PREFIX}{fname}"
        s3.upload_file(os.path.join(LOCAL_MODEL_DIR, fname), SPACES_BUCKET, key)
        print("Wysłano do Spaces:", key)
else:
    print("Pominięto upload do Spaces (brak kluczy w .env)")

## 7. Szybki test modelu

In [ ]:
def predict_time(plec, wiek, czas_5km_s):
    x = pd.DataFrame([{"czas_5km_s": czas_5km_s, "plec": plec, "wiek": wiek}])[FEATURES]
    s = int(final_model.predict(x)[0])
    return f"{s//3600}:{(s%3600)//60:02d}:{s%60:02d}"

print("M, 35 lat, 5 km w 25:00 →", predict_time("M", 35, 25 * 60))
print("K, 28 lat, 5 km w 28:30 →", predict_time("K", 28, 28 * 60 + 30))